#  RAG local completo con interfaz Gradio

**Laboratorio de PLN: Analítica, Textos y Cultura**
Matías Barreto — IFTS24, 2026

**Proyecto integradors**

In [ ]:
%pip install langchain langchain-community langchain-chroma langchain-ollama \
             langchain-text-splitters langchain-core \
             chromadb sentence-transformers \
             gradio pypdf python-dotenv -q

print("✓ Dependencias instaladas")
print("  Recordá tener Ollama corriendo: ollama serve")

  Using cached langchain-1.3.10-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.4.8-py3-none-any.whl.metadata (4.7 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached langgraph-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.8.18-py3-none-any.whl.metadata (21 kB)
  Using cached uuid_utils-0.16.2-cp311-cp311-win_amd64.whl.metad

## ⚙️ Configuración del entorno

Antes de cargar documentos o configurar el LLM, detectamos el hardware disponible.
Eso nos permite elegir el modelo y los parámetros más adecuados sin sobrecalentar la máquina.

In [1]:
import platform
import subprocess

# Detectamos el sistema operativo y la arquitectura del procesador
sistema = platform.system()     # 'Darwin', 'Linux', 'Windows'
maquina = platform.machine()    # 'arm64' (Apple Silicon), 'x86_64'

# Verificamos si hay GPU NVIDIA disponible (solo Linux/Windows)
tiene_cuda = False
if sistema in ("Linux", "Windows"):
    try:
        resultado = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        tiene_cuda = resultado.returncode == 0
    except FileNotFoundError:
        tiene_cuda = False

# Determinamos el perfil de hardware y configuramos el modelo en consecuencia
es_apple_silicon = (sistema == "Darwin" and maquina == "arm64")

if es_apple_silicon:
    PERFIL_HARDWARE = "Apple Silicon (Metal)"
    MODEL_NAME      = "gemma4:e2b"   # liviano, corre bien en M-series
    NUM_CTX         = 4096
elif tiene_cuda:
    PERFIL_HARDWARE = "NVIDIA GPU (CUDA)"
    MODEL_NAME      = "granite4:latest"
    NUM_CTX         = 8192
else:
    PERFIL_HARDWARE = "CPU (sin GPU dedicada)"
    MODEL_NAME      = "gemma4:e2b"   # el más conservador
    NUM_CTX         = 1024

print(f"Sistema:          {sistema} — {maquina}")
print(f"Perfil detectado: {PERFIL_HARDWARE}")
print(f"Modelo elegido:   {MODEL_NAME}")
print(f"Contexto (tokens):{NUM_CTX}")

Sistema:          Windows — AMD64
Perfil detectado: CPU (sin GPU dedicada)
Modelo elegido:   gemma4:e2b
Contexto (tokens):1024


### Importaciones e Ingesta del JSON

In [ ]:
import os
import json
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

#  construimos la ruta al JSON
ruta_base = os.getcwd()
ruta_json = os.path.join(ruta_base, "data", "corpus_limpio.json")

if os.path.exists(ruta_json):
    with open(ruta_json, "r", encoding="utf-8") as archivo:
        mi_corpus = json.load(archivo)
    
    documentos_langchain = [
        Document(page_content=doc["contenido"], metadata=doc["metadata"]) 
        for doc in mi_corpus
    ]
    print(f"[ÉXITO] Se cargaron {len(documentos_langchain)} páginas listas en 'documentos_langchain'.")
else:
    print(f"[ERROR] No se encontró el archivo en: {ruta_json}")

[ÉXITO] Se cargaron 209 páginas listas en 'documentos_langchain'.


### Configuración del RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuración estratégica de los fragmentos
CHUNK_SIZE = 1000      # Tamaño del fragmento 
CHUNK_OVERLAP = 200    # Solapamiento 

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

# procesamos los documentos para generar fragmentos (chunks)
documentos_fragmentados = splitter.split_documents(documentos_langchain)

print(f"--- ESTADÍSTICAS DEL CHUNKING ---")
print(f"Número original de páginas: {len(documentos_langchain)}")
print(f"Número total de fragmentos (chunks) generados: {len(documentos_fragmentados)}")
print(f"Ejemplo del primer fragmento generado:\n")
print(documentos_fragmentados[0].page_content)
print(f"\nMetadata asociada: {documentos_fragmentados[0].metadata}")

--- ESTADÍSTICAS DEL CHUNKING ---
Número original de páginas: 209
Número total de fragmentos (chunks) generados: 586
Ejemplo del primer fragmento generado:

Este documento fue coordinado por Álvaro Soto, Rodrigo Durán, Antonia Moreno y Sebastián Adasme, del Centro Nacional de Inteligencia Artificial (CENIA) de Chile, y Sebastián Rovira, Valeria Jordán y Laura Poveda, funcionarios de la División de Desarrollo Productivo y Empresarial de la Comisión Económica para América Latina y el Caribe (CEPAL). En su elaboración participaron Rodrigo Oportot y Verona Lesseigneur, del CENIA, quienes contaron con el apoyo de Demetris Herakleous, Francisca Lira, funcionarios, y Tomás Rodrigues, Consultor, de la CEPAL, y Salma Jalife, Alberto Farca y Susana Cruz, del Centro México Digital. El documento se elaboró en el marco de la Alianza Digital Unión Europea-América Latina y el Caribe y contó con el financiamiento de la Unión Europea, a través de la estrategia Global Gateway. Ni la Unión Europea ni nin

### Configuración del Modelo de Embedding (Multilingüe)

In [ ]:
import os
#from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings


print("[INFO] Cargando el modelo multilingüe en memoria (esto puede demorar unos segundos)...")

# Inicializamos el modelo de Hugging Face
nombre_modelo = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {'device': 'cpu'} 
encode_kwargs = {'normalize_embeddings': False} 
embeddings_model = HuggingFaceEmbeddings(
    model_name=nombre_modelo,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("[OK] Modelo de embeddings cargado y listo para vectorizar.")

[INFO] Cargando el modelo multilingüe en memoria (esto puede demorar unos segundos)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[OK] Modelo de embeddings cargado y listo para vectorizar.


### Cerrar la conexión para poder borrar la db

In [14]:
# Forzamos la desconexión del cliente si quedó cargado en memoria
if 'vector_store' in locals():
    vector_store = None

import gc
gc.collect()  # El recolector de basura de Python limpia los residuos de memoria
print("[OK] Conexiones liberadas de la memoria.")

[OK] Conexiones liberadas de la memoria.


### borrado de la base de datos 

In [4]:
import shutil

# Definimos la ruta de la base de datos
ruta_vector_db = os.path.join(os.getcwd(), "chroma_db")

# Si la carpeta existe, la borramos por completo
if os.path.exists(ruta_vector_db):
    shutil.rmtree(ruta_vector_db)
    print("[OK] Carpeta 'chroma_db' eliminada para limpiar duplicados anteriores.")
else:
    print("[INFO] No había base de datos previa. Iniciando desde cero.")

[OK] Carpeta 'chroma_db' eliminada para limpiar duplicados anteriores.


### Creación de la Base de Datos Vectorial (ChromaDB Local)

In [ ]:
from langchain_community.vectorstores import Chroma

# ruta donde se guarda la base de datos 
ruta_vector_db = os.path.join(ruta_base, "chroma_db")

print("[INFO] Indexando fragmentos en ChromaDB (calculando similitud coseno)...")

# Creamos la base de datos y guardamos los vectores
vector_store = Chroma.from_documents(
    documents=documentos_fragmentados,
    embedding=embeddings_model,
    persist_directory=ruta_vector_db,
    collection_metadata={"hnsw:space": "cosine"} # Obligamos a usar similitud coseno
)

print(f"[ÉXITO] Base de datos vectorial creada y guardada en de forma persistente en: {ruta_vector_db}")
print(f"  Fragmentos indexados: {len(documentos_fragmentados)}")

C:\Users\admin\AppData\Local\Temp\ipykernel_17668\3033003431.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


[INFO] Indexando fragmentos en ChromaDB (calculando similitud coseno)...
[ÉXITO] Base de datos vectorial creada y guardada en de forma persistente en: d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\chroma_db
  Fragmentos indexados: 586


In [ ]:
# Transformamos la base de datos vectorial en un componente de recuperación (Retriever)
# K=4 fragmentos
retriever_basico = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("[OK] Retriever básico inicializado. Configurado para recuperar los K=4 fragmentos más cercanos.")

[OK] Retriever básico inicializado. Configurado para recuperar los K=4 fragmentos más cercanos.


### Primer test

**vamos a hacer una consulta sobre sostenibilidad**

In [ ]:
query = "¿Cuáles son los principales desafíos del cambio climático mencionados en los informes?"

# busca los vectores en base al ángulo
fragmentos_recuperados = retriever_basico.invoke(query)

print(f"Pregunta realizada: '{query}'\n")
print(f"Se recuperaron {len(fragmentos_recuperados)} fragmentos relevantes.\n")
print("="*60)

# mostramos lo que recuperó
for i, doc in enumerate(fragmentos_recuperados, 1):
    print(f"\n[FRAGMENTO {i}]")
    print(f"Fuente: {doc.metadata.get('fuente', 'Desconocida')} | Página: {doc.metadata.get('pagina', 'N/A')}")
    print(f"Contenido (primeros 300 caracteres):\n{doc.page_content[:300]}...")
    print("-"*60)

# mostramos los caracteres exactos para ver si cambian
for i, doc in enumerate(fragmentos_recuperados, 1):
    print(f"\n[FRAGMENTO {i}]")
    print(f"Fuente: {doc.metadata.get('fuente')} | Página: {doc.metadata.get('pagina')}")
    print(f"Contenido Real:\n{doc.page_content}") 
    print("="*40)

Pregunta realizada: '¿Cuáles son los principales desafíos del cambio climático mencionados en los informes?'

Se recuperaron 4 fragmentos relevantes.


[FRAGMENTO 1]
Fuente: CEPAL | Página: 163
Contenido (primeros 300 caracteres):
. Modelos de entrenamiento intensivo, centros de datos de alto consumo energético y dispositivos de corta vida útil tienen una huella de carbono significativa. Sin políticas que articulen sustentabilidad, eficiencia energética y economía circular, la expansión de la IA puede entrar en contradicción ...
------------------------------------------------------------

[FRAGMENTO 2]
Fuente: CEPAL | Página: 183
Contenido (primeros 300 caracteres):
. Además, este reconocimiento crea estímulos financieros directos, lo que favorece la adopción de certificaciones ambientales. Así, la inclusión pública del código 6311 es un diferenciador claro en el liderazgo regional hacia la infraestructura digital sostenible. CONCLUSIONES03 La sostenibilidad de...
--------------------

### Búsqueda con Filtros Avanzados (Metadata Filtering)

**LangChain permite pasarle condiciones a ChromaDB a través del parámetro search_kwargs.**

In [ ]:
query_filtrada = "Estrategias de mitigación y transición energética"

# Configuramos un filtro: obligamos a que la 'fuente' sea exactamente 'CEPAL'
configuracion_filtros = {
    "k": 3,
    "filter": {"fuente": "CEPAL"} 
}

# Inicializamos un nuevo retriever con esta configuración
retriever_filtrado = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs=configuracion_filtros
)

# Ejecutamos la consulta
fragmentos_filtrados = retriever_filtrado.invoke(query_filtrada)

print(f"Pregunta: '{query_filtrada}' (Filtrado solo para CEPAL)\n")
print(f"Se recuperaron {len(fragmentos_filtrados)} fragmentos.\n")
print("="*60)

for i, doc in enumerate(fragmentos_filtrados, 1):
    print(f"\n[FRAGMENTO {i}]")
    print(f"Fuente Real: {doc.metadata.get('fuente')} | Año: {doc.metadata.get('anio', 'N/A')}")
    print(f"Contenido:\n{doc.page_content[:250]}...")
    print("-"*60)

Pregunta: 'Estrategias de mitigación y transición energética' (Filtrado solo para CEPAL)

Se recuperaron 3 fragmentos.


[FRAGMENTO 1]
Fuente Real: CEPAL | Año: 2026
Contenido:
. Modelos de entrenamiento intensivo, centros de datos de alto consumo energético y dispositivos de corta vida útil tienen una huella de carbono significativa. Sin políticas que articulen sustentabilidad, eficiencia energética y economía circular, la...
------------------------------------------------------------

[FRAGMENTO 2]
Fuente Real: CEPAL | Año: 2026
Contenido:
y un tercero que mide la proporción de energías renovables no convencionales (ERNC) dentro de la matriz energética. Con estas modificaciones, el ILIA 2025 busca entregar información comparativa respecto a temas que son de creciente interés dentro de ...
------------------------------------------------------------

[FRAGMENTO 3]
Fuente Real: CEPAL | Año: 2026
Contenido:
. Los primeros resultados globales son alentadores: Green Light ha logrado redu

### Búsqueda por Umbral de Similitud (Similarity Score Threshold)

**Para evitar que el sistema procese basura, podemos configurar un retriever que solo devuelva fragmentos si superan un umbral mínimo de similitud (56 a 62)**

In [ ]:
# Configuramos el retriever para que filtre
retriever_umbral = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.62 # Exigimos al menos 62% de similitud semántica
    }
)

# Probamos con una pregunta relevante
print("--- TEST 1: Pregunta coherente ---")
docs_validos = retriever_umbral.invoke("Desarrollo sostenible y metas para el 2030")
print(f"Fragmentos recuperados para pregunta válida: {len(docs_validos)}")

# Probamos con una pregunta completamente fuera de contexto
print("\n--- TEST 2: Pregunta fuera de contexto ---")
docs_fuera = retriever_umbral.invoke("¿Cuál es la alineación de la selección argentina?")
print(f"Fragmentos recuperados para pregunta fuera de contexto: {len(docs_fuera)}")

No relevant docs were retrieved using the relevance score threshold 0.56


--- TEST 1: Pregunta coherente ---
Fragmentos recuperados para pregunta válida: 3

--- TEST 2: Pregunta fuera de contexto ---
Fragmentos recuperados para pregunta fuera de contexto: 0


### LLM local con Ollama

**Usamos el modelo que detectamos al inicio. El parámetro `num_ctx` le dice a Ollama cuántos tokens puede usar como contexto — más contexto permite fragmentos más largos pero requiere más memoria.**

In [10]:
from langchain_ollama import OllamaLLM

# Conectamos con el servidor Ollama que corre localmente en el puerto 11434
# temperature=0.1 para respuestas precisas y reproducibles
llm = OllamaLLM(
    model=MODEL_NAME,
    temperature=0.1,
    num_ctx=NUM_CTX,
)

# Verificamos que Ollama responde antes de armar el pipeline completo
respuesta_prueba = llm.invoke("Respondé solo con 'ok' si estás funcionando.")
print(f"✓ Ollama responde: {respuesta_prueba.strip()}")
print(f"  Modelo activo: {MODEL_NAME}")

✓ Ollama responde: ok
  Modelo activo: gemma4:e2b


### Pipeline RAG con LCEL

Ensamblamos el pipeline completo usando LCEL (LangChain Expression Language). La cadena conecta retriever → prompt → LLM → parser en una sola expresión que podemos invocar con `.invoke(pregunta)`.

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# El retriever busca los 3 fragmentos más similares a la pregunta
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def formatear_documentos(docs):
    # Une los fragmentos recuperados en un solo bloque de texto
    return "\n\n".join(doc.page_content for doc in docs)

TEMPLATE = """Sos un asistente que responde preguntas basándose ÚNICAMENTE en los documentos proporcionados.
Si la respuesta no está en los documentos, decilo claramente: no la inventes.

Documentos:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ Pipeline RAG configurado")
print("  Flujo: pregunta → ChromaDB (k=3) → prompt → Ollama → respuesta")

✓ Pipeline RAG configurado
  Flujo: pregunta → ChromaDB (k=3) → prompt → Ollama → respuesta


### Probamos el pipeline con una pregunta de prueba antes de abrir la interfaz

In [13]:
pregunta_test = "¿De qué tratan los documentos que cargaste?"
pregunta_test = "¿Qué metas o desafíos principales se plantean para el desarrollo sostenible?"

respuesta = pipeline_rag.invoke(pregunta_test)

print(f"Pregunta: {pregunta_test}")
print(f"\nRespuesta:")
print(respuesta)

Pregunta: ¿Qué metas o desafíos principales se plantean para el desarrollo sostenible?

Respuesta:
La región posee oportunidades únicas para un desarrollo sostenible de infraestructuras digitales. El objetivo es identificar datos específicos de sostenibilidad asociados a centros de datos para reconocer brechas y mejores prácticas en materia de infraestructura verde para IA, complementando la visión integral de un desarrollo de la IA para América Latina y el Caribe que sea ético y sostenible. Además, la gobernanza es un elemento fundamental para el desarrollo sustentable y armónico de los ecosistemas de IA.


## GRADIO

In [19]:
def responder_rag(mensaje, historial):
    try:
        print(f"[INFO] Procesando consulta: '{mensaje}'")
        
        # 1. Recuperamos los fragmentos para guardarlos en el registro de la interfaz
        documentos_recuperados = retriever_basico.invoke(mensaje)
        
        # 2. Construimos una cadena de texto para mostrarle al usuario de dónde salió la info
        fuentes_usadas = "\n\n--- 📄 FUENTES RECOVERY ---"
        for i, doc in enumerate(documentos_recuperados, 1):
            fuentes_usadas += f"\n• [{doc.metadata.get('fuente')} - Pág. {doc.metadata.get('pagina')}]"
        
        # 3. Ejecutamos la respuesta del modelo local
        respuesta_llm = pipeline_rag.invoke(mensaje)
        
        # 4. Devolvemos la respuesta combinada: texto del modelo + las fuentes reales
        return f"{respuesta_llm}\n\n{fuentes_usadas}"
        
    except Exception as e:
        return f"Error al procesar la respuesta con el modelo local: {str(e)}"

In [20]:
import gradio as gr

# Configuramos la estructura de la página de forma limpia
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Asistente RAG - Informes de Sostenibilidad")
    gr.Markdown("Consultá de forma inteligente los documentos de la CEPAL y el BID mediante embeddings y modelos locales.")
    
    # Creamos la interfaz del chat simplificada (Gradio maneja los botones por defecto)
    chatbot = gr.ChatInterface(
        fn=responder_rag,
        textbox=gr.Textbox(
            placeholder="Escribí tu consulta sobre sustentabilidad o transición energética...", 
            container=False, 
            scale=7
        ),
        examples=[
            "¿Qué se menciona sobre las energías renovables?",
            "¿Cuáles son los desafíos del cambio climático?",
            "¿en que año fue aprobado por las Naciones Unidas el Pacto Digital Global?"
        ]
    )

# Lanzamos la aplicación
if __name__ == "__main__":
    demo.launch(share=False)

C:\Users\admin\AppData\Local\Temp\ipykernel_17668\591130202.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[INFO] Procesando consulta: '¿qué pais invierte mas en IA en America Latina?'


d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[INFO] Procesando consulta: '¿qué pais invierte mas en IA en América Latina?'


d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
